In [1]:
import pandas as pd
import numpy as np

In [2]:
# Carregar os dados
# Códigos IBGE dos distritos (e bairros, onde aplicável) do estado
demanda_df = pd.read_csv('Documents/SP_coords_medias_divisoes.csv')

# Códigos IBGE dos estabelecimentos (hospitais gerais)
facilidades_df = pd.read_csv('Documents/Hospitais_gerais_SP_CNES.csv')

In [3]:
# --- PARÂMETRO DE CORREÇÃO ---
FATOR_CIRCUIDADE = 1.3  # Multiplicador para estimar distância real em vias

# --- FUNÇÃO ATUALIZADA (com o fator aplicado no return) ---
def haversine_vectorized(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1[:, np.newaxis]
    dlon = lon2 - lon1[:, np.newaxis]
    a = np.sin(dlat/2)**2 + np.cos(lat1[:, np.newaxis]) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    # Retorna a distância em km já corrigida pelo fator de circuidade
    return (R * c) * FATOR_CIRCUIDADE

In [4]:
# 1. Preparar IDs e Coordenadas
ids_demanda = demanda_df['ID_divisao'].astype(str).tolist()
ids_fac = facilidades_df['CO_UNIDADE'].astype(str).tolist()

# Criar os nomes conforme seu padrão
labels_demanda = [f"D_{id_}" for id_ in ids_demanda]
labels_facilidade = [f"F_{id_}" for id_ in ids_fac]

# 2. Extrair coordenadas como arrays do NumPy
lat_dem = demanda_df['MEAN_Y'].values
lon_dem = demanda_df['MEAN_X'].values

# O destino M é a união dos pontos de demanda (3146) + facilidades (960)
lat_alvo = np.concatenate([lat_dem, facilidades_df['NU_LATITUDE'].values])
lon_alvo = np.concatenate([lon_dem, facilidades_df['NU_LONGITUDE'].values])
labels_alvo = labels_demanda + labels_facilidade

# 3. Calcular a matriz (Resultado em Quilômetros)
print(f"Calculando matriz {len(labels_demanda)} x {len(labels_alvo)}...")
matriz_dist = haversine_vectorized(lat_dem, lon_dem, lat_alvo, lon_alvo)

# 4. Transformar em DataFrame com cabeçalhos e índices
A_total_df = pd.DataFrame(matriz_dist, index=labels_demanda, columns=labels_alvo)

# Salvar para não perder o progresso
A_total_df.to_csv('matriz_distancia_SP_haversine.csv')
print("Matriz concluída e salva!")

Calculando matriz 3146 x 4105...
Matriz concluída e salva!
